In [1]:
# for data manipulation
import pandas as pd

import sklearn

# for data preprocessing and pipeline creation
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import make_column_transformer
from sklearn.pipeline import make_pipeline

# for model training, tuning, and evaluation
import xgboost as xgb
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import accuracy_score, classification_report, recall_score

# for model serialization
import joblib

# for creating a folder
import os

# for hugging face space authentication to upload files
from huggingface_hub import login, HfApi

In [2]:
# Set scikit-learn's display mode to 'diagram' for better visualization of pipelines and estimators
sklearn.set_config(display='diagram')

## Data Loading and Overview

In [3]:
# Load the dataset from a CSV file into a Pandas DataFrame
bank_churn = pd.read_csv(r"C:\Users\New\Desktop\Jupyter Notebook\Data Science Project\bank_customer_churn.csv")

In [4]:
# Create a copy of the dataframe
dataset = bank_churn.copy()

In [5]:
# Display the first five rows of the dataset
dataset.head()

,CustomerId,Surname,CreditScore,Geography,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited
0,15634602,Hargrave,619,France,42.0,2,0.00,1,1.0,1.0,101348.88,1
1,15647311,Hill,608,Spain,41.0,1,83807.86,1,0.0,1.0,112542.58,0
2,15619304,Onio,502,France,42.0,8,159660.80,3,1.0,0.0,113931.57,1
3,15701354,Boni,699,France,39.0,1,0.00,2,0.0,0.0,93826.63,0
4,15737888,Mitchell,850,Spain,43.0,2,125510.82,1,NaN,1.0,79084.10,0


In [6]:
# Display the number of rows and columns in the dataset
dataset.shape

(10002, 12)

In [7]:
# Display the column names of the dataset
dataset.columns

Index(['CustomerId', 'Surname', 'CreditScore', 'Geography', 'Age', 'Tenure',
       'Balance', 'NumOfProducts', 'HasCrCard', 'IsActiveMember',
       'EstimatedSalary', 'Exited'],
      dtype='str')

## EDA

In [8]:
# Define the target variable for the classification task
target = 'Exited'

# List of numerical features in the dataset
numeric_features = [
    'CreditScore',       # Customer's credit score
    'Age',               # Customer's age
    'Tenure',            # Number of years the customer has been with the bank
    'Balance',           # Customer’s account balance
    'NumOfProducts',     # Number of products the customer has with the bank
    'HasCrCard',         # Whether the customer has a credit card (binary: 0 or 1)
    'IsActiveMember',    # Whether the customer is an active member (binary: 0 or 1)
    'EstimatedSalary'    # Customer’s estimated salary
]

# List of categorical features in the dataset
categorical_features = [
    'Geography',         # Country where the customer resides
]

In [9]:
# Generate summary statistics for numerical features
dataset[numeric_features].describe()

,CreditScore,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary
count,10002.000000,10001.000000,10002.000000,10002.000000,10002.000000,10001.000000,10001.000000,10002.000000
mean,650.555089,38.922311,5.012498,76491.112875,1.530194,0.705529,0.514949,100083.331145
std,96.661615,10.487200,2.891973,62393.474144,0.581639,0.455827,0.499801,57508.117802
min,350.000000,18.000000,0.000000,0.000000,1.000000,0.000000,0.000000,11.580000
25%,584.000000,32.000000,3.000000,0.000000,1.000000,0.000000,0.000000,50983.750000
50%,652.000000,37.000000,5.000000,97198.540000,1.000000,1.000000,1.000000,100185.240000
75%,718.000000,44.000000,7.000000,127647.840000,2.000000,1.000000,1.000000,149383.652500
max,850.000000,92.000000,10.000000,250898.090000,4.000000,1.000000,1.000000,199992.480000


1. Balance Distribution:

The mean balance is ~76,491, but the 25th percentile is 0, indicating that at least 25% of customers have a balance of zero.
Possible explanations:
Customers may use the account primarily for transactions, withdrawing funds immediately after deposits.
Some customers might have secondary accounts with zero balance.
Certain customers may have no savings account but only hold credit cards.

2. Age and Credit Score:

The average age is ~39 years, with a max of 92 years.
The credit score ranges from 350 to 850, with an average of 650.


3. Products and Activity:

Most customers have 1 or 2 products (median = 1).
Around 70% have a credit card, and 51% are active members.


4. Churn Rate:

The Exited (churn) rate is ~20%, meaning 1 in 5 customers leave the bank.